# Bibliotecas
 - Coleta de dados validados como boatos = falso
 - Usa a Google Fact Check Tools API para buscar afirmações já verificadas
 - Suporta MODO_TESTE para validação rápida antes da coleta completa

In [1]:
import os
import time
import random
import requests
import pandas as pd

from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv

# Google Fact Check

## Configuração e modo de execução

**MODO_TESTE = True** → roda apenas os primeiros `N_TERMOS_TESTE` termos com no máximo
`MAX_PAGINAS_TESTE` páginas cada. Use para validar que a API está respondendo antes
de executar a coleta completa.

**MODO_TESTE = False** → coleta completa com todos os termos e configurações de produção.

In [2]:
# ==============================================================
# MODO DE EXECUÇÃO — altere aqui antes de rodar
# ==============================================================
MODO_TESTE        = False  # True = teste rápido | False = coleta completa
N_TERMOS_TESTE    = 5     # quantos termos usar no modo teste
MAX_PAGINAS_TESTE = 2     # máximo de páginas por termo no modo teste

# ==============================================================
# CHAVE DA API
# ==============================================================
load_dotenv("../google-factcheck-api-key.env")

API_KEY = os.getenv("GOOGLE_FACTCHECK_API_KEY")

if not API_KEY:
    raise ValueError("Chave da API não encontrada. Verifique o arquivo google-factcheck-api-key.env")

# ==============================================================
# PARÂMETROS DA API — produção
# ==============================================================
URL          = "https://factchecktools.googleapis.com/v1alpha1/claims:search"
PAGE_SIZE    = 50    # máximo permitido pela API
MAX_AGE_DAYS = 1825  # 5 anos de histórico
MAX_PAGINAS  = 10    # máximo de páginas por termo em produção

# ==============================================================
# PARÂMETROS DE RETRY — proteção contra erro 503
# ==============================================================
MAX_TENTATIVAS        = 5  # tentativas por requisição antes de desistir
BACKOFF_BASE_SEGUNDOS = 5  # espera base em segundos (dobra a cada tentativa)
#
# Sequência de espera com jitter ±1s:
#   tentativa 1 →  5s ± 1s  (~4–6s)
#   tentativa 2 → 10s ± 1s  (~9–11s)
#   tentativa 3 → 20s ± 1s  (~19–21s)
#   tentativa 4 → 40s ± 1s  (~39–41s)
#   tentativa 5 → 80s ± 1s  (~79–81s)

# ==============================================================
# INTERVALOS ENTRE REQUISIÇÕES — reduz throttling
# ==============================================================
SLEEP_ENTRE_PAGINAS = 2.0  # segundos entre páginas do mesmo termo
SLEEP_ENTRE_TERMOS  = 5.0  # segundos entre termos diferentes

# Aplica limites do modo teste se ativado
max_paginas_por_termo = MAX_PAGINAS_TESTE if MODO_TESTE else MAX_PAGINAS

print(f"Modo de execução : {'TESTE' if MODO_TESTE else 'PRODUÇÃO'}")
print(f"Max páginas/termo: {max_paginas_por_termo}")
print(f"Max age days     : {MAX_AGE_DAYS}")
print(f"Max tentativas   : {MAX_TENTATIVAS}  (backoff: "
      + ", ".join(f"{BACKOFF_BASE_SEGUNDOS * 2**i}s" for i in range(MAX_TENTATIVAS)) + ")")
print(f"Sleep pág/termo  : {SLEEP_ENTRE_PAGINAS}s / {SLEEP_ENTRE_TERMOS}s")

Modo de execução : PRODUÇÃO
Max páginas/termo: 10
Max age days     : 1825
Max tentativas   : 5  (backoff: 5s, 10s, 20s, 40s, 80s)
Sleep pág/termo  : 2.0s / 5.0s


## Termos de busca

Organizados por categoria para facilitar manutenção e expansão futura.
Cada categoria pode ser desabilitada individualmente antes de uma coleta parcial.

In [3]:
# ==============================================================
# TERMOS POR CATEGORIA
# Comente ou remova categorias inteiras para coletas parciais
# ==============================================================

termos_eleitoral = [
    "urnas eletrônicas",
    "fraude nas urnas",
    "TSE",
    "eleições 2022",
    "eleições 2026",
    "voto impresso",
    "urna auditável",
    "fraude eleitoral",
    "auditoria das urnas",
    "sistema eleitoral brasileiro",
]

termos_figuras_politicas = [
    "Lula",
    "Bolsonaro",
    "Alexandre de Moraes",
    "Moraes",
    "Nikolas Ferreira",
    "Haddad",
    "Erika Hilton",
    "PT",
    "PL",
    "Tarcísio de Freitas",
    "Ciro Gomes",
    "Flávio Bolsonaro",
    "Gilmar Mendes",
    "Marina Silva",
    "Gleisi Hoffmann",
    "Arthur Lira",
    "Rodrigo Pacheco",
    "Roberto Campos Neto",
    "Dias Toffoli",
    "Kassio Nunes",
]

termos_institucional = [
    "STF redes sociais",
    "Congresso Nacional",
    "anistia Bolsonaro",
    "PL da anistia",
    "8 de janeiro",
    "golpe de estado",
    "preso político",
    "inquérito das fake news",
    "censura redes sociais",
    "liberdade de imprensa",
]

termos_economico = [
    "INSS",
    "fraude INSS",
    "Pix imposto",
    "taxação do Pix",
    "salário mínimo",
    "imposto de renda",
    "reforma tributária",
    "arcabouço fiscal",
    "Petrobras",
    "Banco Central",
    "inflação",
    "privatização",
    "dívida pública",
    "gastos públicos",
]

termos_saude = [
    "vacinas covid",
    "cloroquina",
    "ivermectina",
    "SUS",
    "dengue",
    "lockdown",
    "passaporte vacinal",
    "mortes covid",
    "kit covid",
]

termos_social_seguranca = [
    "MST",
    "reforma agrária",
    "quilombolas",
    "FUNAI",
    "demarcação de terras",
    "violência policial",
    "milícia",
]

termos_programas_sociais = [
    "Bolsa Família",
    "BPC",
    "Minha Casa Minha Vida",
    "auxílio emergencial",
    "reforma da previdência",
    "desemprego",
    "fome",
]

# ==============================================================
# LISTA UNIFICADA — mantém a ordem das categorias
# ==============================================================
termos_busca = (
    termos_eleitoral
    + termos_figuras_politicas
    + termos_institucional
    + termos_economico
    + termos_saude
    + termos_social_seguranca
    + termos_programas_sociais
)

print(f"Total de termos configurados: {len(termos_busca)}")
print(f"  Eleitoral          : {len(termos_eleitoral)}")
print(f"  Figuras políticas  : {len(termos_figuras_politicas)}")
print(f"  Institucional      : {len(termos_institucional)}")
print(f"  Econômico          : {len(termos_economico)}")
print(f"  Saúde              : {len(termos_saude)}")
print(f"  Social/Segurança   : {len(termos_social_seguranca)}")
print(f"  Programas sociais  : {len(termos_programas_sociais)}")

Total de termos configurados: 77
  Eleitoral          : 10
  Figuras políticas  : 20
  Institucional      : 10
  Econômico          : 14
  Saúde              : 9
  Social/Segurança   : 7
  Programas sociais  : 7


## Seleção de termos conforme modo de execução

In [4]:
if MODO_TESTE:
    # No modo teste, usa apenas os primeiros N termos da lista unificada
    # para validar que a API está respondendo e o pipeline funciona
    termos_para_coletar = termos_busca[:N_TERMOS_TESTE]
    print(f"MODO TESTE — coletando {len(termos_para_coletar)} termos:")
else:
    termos_para_coletar = termos_busca
    print(f"MODO PRODUÇÃO — coletando todos os {len(termos_para_coletar)} termos:")

for t in termos_para_coletar:
    print(f"  - {t}")

MODO PRODUÇÃO — coletando todos os 77 termos:
  - urnas eletrônicas
  - fraude nas urnas
  - TSE
  - eleições 2022
  - eleições 2026
  - voto impresso
  - urna auditável
  - fraude eleitoral
  - auditoria das urnas
  - sistema eleitoral brasileiro
  - Lula
  - Bolsonaro
  - Alexandre de Moraes
  - Moraes
  - Nikolas Ferreira
  - Haddad
  - Erika Hilton
  - PT
  - PL
  - Tarcísio de Freitas
  - Ciro Gomes
  - Flávio Bolsonaro
  - Gilmar Mendes
  - Marina Silva
  - Gleisi Hoffmann
  - Arthur Lira
  - Rodrigo Pacheco
  - Roberto Campos Neto
  - Dias Toffoli
  - Kassio Nunes
  - STF redes sociais
  - Congresso Nacional
  - anistia Bolsonaro
  - PL da anistia
  - 8 de janeiro
  - golpe de estado
  - preso político
  - inquérito das fake news
  - censura redes sociais
  - liberdade de imprensa
  - INSS
  - fraude INSS
  - Pix imposto
  - taxação do Pix
  - salário mínimo
  - imposto de renda
  - reforma tributária
  - arcabouço fiscal
  - Petrobras
  - Banco Central
  - inflação
  - privatiz

## Função de requisição com retry e backoff exponencial

Em caso de erro 503/504/500/429, a função aguarda um tempo crescente antes de
tentar novamente — até `MAX_TENTATIVAS` vezes. Isso evita que termos inteiros
sejam perdidos silenciosamente por instabilidades temporárias da API.

**Sequência de espera** (backoff `5 × 2^(n-1)` + jitter ±1s):

| Tentativa | Espera base | Com jitter |
|-----------|-------------|------------|
| 1 | 5s | ~4–6s |
| 2 | 10s | ~9–11s |
| 3 | 20s | ~19–21s |
| 4 | 40s | ~39–41s |
| 5 | 80s | ~79–81s |

Erros não-recuperáveis (400, 401, 403) abortam imediatamente sem retry.

In [5]:
def requisitar_com_retry(url: str, params: dict) -> requests.Response | None:
    """
    Faz uma requisição GET com até MAX_TENTATIVAS tentativas em erros recuperáveis.

    Backoff: espera = BACKOFF_BASE_SEGUNDOS * 2^(tentativa-1) + jitter ±1s
      tentativa 1 →  5s ± 1s
      tentativa 2 → 10s ± 1s
      tentativa 3 → 20s ± 1s
      tentativa 4 → 40s ± 1s
      tentativa 5 → 80s ± 1s

    Retorna o Response em caso de sucesso (HTTP 200).
    Retorna None se todas as tentativas falharem por erro recuperável.
    Retorna o Response imediatamente em erros não-recuperáveis (400, 401, 403).
    """
    ERROS_RECUPERAVEIS = {429, 500, 503, 504}

    for tentativa in range(1, MAX_TENTATIVAS + 1):

        resposta = None
        try:
            resposta = requests.get(url, params=params, timeout=30)
        except requests.exceptions.RequestException as exc:
            print(f"    [tentativa {tentativa}/{MAX_TENTATIVAS}] Erro de conexão: {exc}")

        # Sucesso
        if resposta is not None and resposta.status_code == 200:
            return resposta

        status = resposta.status_code if resposta is not None else "conexão falhou"

        # Erro não-recuperável — não adianta tentar de novo
        if resposta is not None and resposta.status_code not in ERROS_RECUPERAVEIS:
            print(f"    Erro não-recuperável: HTTP {status} (sem retry)")
            return resposta

        # Erro recuperável — calcula espera e tenta de novo (se ainda há tentativas)
        if tentativa < MAX_TENTATIVAS:
            espera_base = BACKOFF_BASE_SEGUNDOS * (2 ** (tentativa - 1))
            jitter      = random.uniform(-1.0, 1.0)
            espera_total = round(max(1.0, espera_base + jitter), 1)
            print(
                f"    [tentativa {tentativa}/{MAX_TENTATIVAS}] HTTP {status} "
                f"— aguardando {espera_total}s antes de tentar novamente..."
            )
            time.sleep(espera_total)
        else:
            print(
                f"    [tentativa {tentativa}/{MAX_TENTATIVAS}] HTTP {status} "
                f"— todas as tentativas esgotadas."
            )

    # Todas as tentativas falharam
    return None


print("Função de retry definida (5 tentativas, backoff 5s→10s→20s→40s→80s ± 1s).")

Função de retry definida (5 tentativas, backoff 5s→10s→20s→40s→80s ± 1s).


## Coleta — API Google Fact Check

In [6]:
registros            = []
termos_com_resultado = []   # termos que retornaram ao menos 1 claim
termos_sem_resultado = []   # termos que retornaram 0 claims sem erro
termos_com_erro      = []   # termos que falharam mesmo após todos os retries
detalhes_erros       = []   # log detalhado de cada falha

total_termos = len(termos_para_coletar)

for idx, termo in enumerate(termos_para_coletar, start=1):

    print(f"\n[{idx}/{total_termos}] Consultando: {termo}")

    page_token   = None
    pagina_atual = 1
    claims_termo = 0
    erro_neste_termo = False

    while pagina_atual <= max_paginas_por_termo:

        params = {
            "query"       : termo,
            "languageCode": "pt",
            "pageSize"    : PAGE_SIZE,
            "maxAgeDays"  : MAX_AGE_DAYS,
            "key"         : API_KEY,
        }

        if page_token:
            params["pageToken"] = page_token

        resposta = requisitar_com_retry(URL, params)

        # Falha persistente após todos os retries
        if resposta is None or resposta.status_code != 200:
            status_code = resposta.status_code if resposta is not None else "conexão falhou"
            mensagem    = ""
            if resposta is not None:
                try:
                    mensagem = resposta.json().get("error", {}).get("message", "")
                except Exception:
                    mensagem = resposta.text[:200]

            print(f"  ✗ Abortando '{termo}' (HTTP {status_code}, pág. {pagina_atual})")

            termos_com_erro.append(termo)
            detalhes_erros.append({
                "termo"      : termo,
                "pagina"     : pagina_atual,
                "status_code": status_code,
                "mensagem"   : mensagem,
                "horario"    : datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
            })
            erro_neste_termo = True
            break

        dados  = resposta.json()
        claims = dados.get("claims", [])

        print(f"  Página {pagina_atual} — {len(claims)} claims")
        claims_termo += len(claims)

        for claim in claims:
            texto_claim = claim.get("text", "")
            data_claim  = claim.get("claimDate", "")

            for review in claim.get("claimReview", []):
                registros.append({
                    "termo_busca"       : termo,
                    "texto_afirmacao"   : texto_claim,
                    "data_claim"        : data_claim,
                    "fonte_verificacao" : review.get("publisher", {}).get("name", ""),
                    "url_checagem"      : review.get("url", ""),
                    "avaliacao_original": review.get("textualRating", ""),
                    "data_publicacao"   : review.get("reviewDate", ""),
                    "url_consulta"      : resposta.url,
                    "data_coleta"       : datetime.now().strftime("%d/%m/%Y %H:%M:%S"),
                    "origem_pipeline"   : "GOOGLE_FACTCHECK",
                })

        page_token = dados.get("nextPageToken")
        if not page_token:
            break

        pagina_atual += 1
        time.sleep(SLEEP_ENTRE_PAGINAS)

    # Classifica o termo conforme resultado
    if not erro_neste_termo:
        if claims_termo > 0:
            termos_com_resultado.append(termo)
        else:
            termos_sem_resultado.append(termo)
            print(f"  — Nenhuma claim retornada para '{termo}'")

    # Pausa entre termos (somente se não for o último)
    if idx < total_termos:
        time.sleep(SLEEP_ENTRE_TERMOS)

# ==============================================================
# RESUMO FINAL DA COLETA
# ==============================================================
separador = "=" * 56
print(f"\n{separador}")
print(f"  RESUMO DA COLETA — {'TESTE' if MODO_TESTE else 'PRODUÇÃO'}")
print(separador)
print(f"  Total de registros brutos coletados : {len(registros)}")
print(f"  Termos com resultado (≥1 claim)     : {len(termos_com_resultado)}")
print(f"  Termos sem resultado (0 claims)     : {len(termos_sem_resultado)}")
print(f"  Termos com erro (falha persistente) : {len(termos_com_erro)}")
print(separador)

if termos_sem_resultado:
    print(f"\n  Termos sem resultado:")
    for t in termos_sem_resultado:
        print(f"    - {t}")

if termos_com_erro:
    print(f"\n  Termos com erro — rode novamente ou ajuste a lista:")
    for d in detalhes_erros:
        print(f"    - {d['termo']:35s}  HTTP {d['status_code']}  pág.{d['pagina']}  {d['horario']}")

print(f"\n{separador}")


[1/77] Consultando: urnas eletrônicas
  Página 1 — 50 claims
  Página 2 — 50 claims
  Página 3 — 18 claims

[2/77] Consultando: fraude nas urnas
  Página 1 — 50 claims
  Página 2 — 35 claims

[3/77] Consultando: TSE
  Página 1 — 50 claims
  Página 2 — 50 claims
  Página 3 — 0 claims

[4/77] Consultando: eleições 2022
  Página 1 — 27 claims

[5/77] Consultando: eleições 2026
  Página 1 — 18 claims

[6/77] Consultando: voto impresso
  Página 1 — 40 claims

[7/77] Consultando: urna auditável
  Página 1 — 4 claims

[8/77] Consultando: fraude eleitoral
  Página 1 — 35 claims

[9/77] Consultando: auditoria das urnas
  Página 1 — 19 claims

[10/77] Consultando: sistema eleitoral brasileiro
  Página 1 — 43 claims

[11/77] Consultando: Lula
  Página 1 — 50 claims
  Página 2 — 50 claims
  Página 3 — 50 claims
  Página 4 — 50 claims
  Página 5 — 50 claims
  Página 6 — 50 claims
  Página 7 — 50 claims
  Página 8 — 50 claims
  Página 9 — 50 claims
  Página 10 — 0 claims

[12/77] Consultando: Bolso

## DataFrame raw e deduplicação interna

Deduplica dentro desta coleta antes de salvar.
A deduplicação entre coletas diferentes é responsabilidade do notebook de curadoria.

In [7]:
df_google_raw = pd.DataFrame(registros)

print(f"Registros brutos coletados nesta execução: {len(df_google_raw)}")

if len(df_google_raw) > 0:
    # Deduplicação dentro desta coleta:
    # mesma afirmação verificada pela mesma fonte = registro duplicado
    antes = len(df_google_raw)
    df_google_raw = df_google_raw.drop_duplicates(
        subset=["texto_afirmacao", "fonte_verificacao"],
        keep="first"
    ).reset_index(drop=True)
    print(f"Duplicatas removidas nesta coleta  : {antes - len(df_google_raw)}")
    print(f"Registros únicos para salvar       : {len(df_google_raw)}")

    print(f"\nDistribuição por avaliacao_original (top 10):")
    print(df_google_raw["avaliacao_original"].value_counts().head(10))

    print(f"\nDistribuição por fonte_verificacao (top 10):")
    print(df_google_raw["fonte_verificacao"].value_counts().head(10))

    display(df_google_raw.head(3))
else:
    print("Nenhum registro coletado. Verifique a chave da API e os termos de busca.")

Registros brutos coletados nesta execução: 5209
Duplicatas removidas nesta coleta  : 1771
Registros únicos para salvar       : 3438

Distribuição por avaliacao_original (top 10):
avaliacao_original
Falso               1427
falso                788
Enganoso             575
Errado               101
não_é_bem_assim       91
Distorcido            70
Fora de contexto      47
Insustentável         37
Sem contexto          28
verdadeiro            17
Name: count, dtype: int64

Distribuição por fonte_verificacao (top 10):
fonte_verificacao
Aos Fatos           915
Estadão             736
UOL Notícias        527
AFP Checamos        516
Boatos.org          286
Projeto Comprova    224
Observador          118
Folha - UOL          55
BOL - UOL            42
Metrópoles            7
Name: count, dtype: int64


,termo_busca,texto_afirmacao,data_claim,fonte_verificacao,url_checagem,avaliacao_original,data_publicacao,url_consulta,data_coleta,origem_pipeline
0,urnas eletrônicas,"Pilili, mascote da urna eletrônica, faz o L de...",,BOL - UOL,https://www.bol.uol.com.br/noticias/2026/05/08...,Falso,,https://factchecktools.googleapis.com/v1alpha1...,17/05/2026 02:30:46,GOOGLE_FACTCHECK
1,urnas eletrônicas,"Pilili, mascote da urna eletrônica, faz o L de...",,UOL Notícias,https://noticias.uol.com.br/confere/ultimas-no...,Falso,,https://factchecktools.googleapis.com/v1alpha1...,17/05/2026 02:30:46,GOOGLE_FACTCHECK
2,urnas eletrônicas,Lula perdeu todas as eleições com votação manu...,2025-12-07T00:00:00Z,AFP Checamos,https://checamos.afp.com/doc.afp.com.88868T3,Enganoso,2025-12-15T17:22:00Z,https://factchecktools.googleapis.com/v1alpha1...,17/05/2026 02:30:46,GOOGLE_FACTCHECK


## Extração — salvar CSV raw com timestamp

In [8]:
nome_pipeline = "pipeline_falso_google_factcheck"
nome_base     = "google_factcheck"
sufixo_modo   = "_TESTE" if MODO_TESTE else ""
data_agora    = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
pasta_raw.mkdir(parents=True, exist_ok=True)

# ==============================================================
# 1. CSV PRINCIPAL — registros coletados
# ==============================================================
if len(df_google_raw) == 0:
    print("Nenhum registro para salvar. Abortando exportação do CSV principal.")
else:
    caminho_principal = pasta_raw / f"{nome_base}_raw{sufixo_modo}_{data_agora}.csv"
    df_google_raw.to_csv(caminho_principal, index=False, encoding="utf-8-sig")

    print(f"CSV principal salvo : {caminho_principal}")
    print(f"Registros exportados: {len(df_google_raw)}")
    print(f"Modo                : {'TESTE' if MODO_TESTE else 'PRODUÇÃO'}")
    print(f"Data/hora           : {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

# ==============================================================
# 2. CSV DE LOG DE ERROS — apenas se houve falhas
# ==============================================================
if detalhes_erros:
    df_erros = pd.DataFrame(detalhes_erros)
    caminho_erros = pasta_raw / f"{nome_base}_erros{sufixo_modo}_{data_agora}.csv"
    df_erros.to_csv(caminho_erros, index=False, encoding="utf-8-sig")

    print(f"\nCSV de erros salvo  : {caminho_erros}")
    print(f"Termos com erro     : {len(df_erros)}")
    print("\nConteúdo do log de erros:")
    print(df_erros[["termo", "pagina", "status_code", "horario"]].to_string(index=False))
else:
    print("\nNenhum erro registrado — CSV de log de erros não gerado.")

# ==============================================================
# 3. INSTRUÇÃO FINAL
# ==============================================================
if MODO_TESTE:
    print(f"\n{'=' * 56}")
    print(f"  MODO TESTE concluído.")
    print(f"  Para rodar a coleta completa:")
    print(f"    1. Defina MODO_TESTE = False na célula de configuração")
    print(f"    2. Execute o notebook do início")
    if detalhes_erros:
        print(f"\n  Termos com erro para tentar de novo:")
        print(f"    Copie a lista abaixo e use em uma coleta de retry:")
        termos_erro_str = "\n".join(f'    "{t}",' for t in termos_com_erro)
        print(termos_erro_str)
    print(f"{'=' * 56}")

CSV principal salvo : ..\dados\pipeline_falso_google_factcheck\raw\google_factcheck_raw_2026-05-17_02-42-35.csv
Registros exportados: 3438
Modo                : PRODUÇÃO
Data/hora           : 17/05/2026 02:42:35

Nenhum erro registrado — CSV de log de erros não gerado.
